In [ ]:
#|default_exp models.ROCKET

# ROCKET

>ROCKET (RandOm Convolutional KErnel Transform) functions for univariate and multivariate time series.

In [ ]:
#|export
import sklearn
from sklearn.linear_model import RidgeClassifierCV, RidgeCV
from sklearn.metrics import make_scorer
from sklearn.preprocessing import StandardScaler

from tsai.data.external import *
from tsai.imports import *
from tsai.models.layers import *

In [ ]:
#|export
class RocketClassifier(sklearn.pipeline.Pipeline):
    """Time series classification using ROCKET features and a linear classifier"""
    
    def __init__(self, num_kernels=10_000, normalize_input=True, random_state=None, 
                 alphas=(0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0), normalize_features=True, memory=None, verbose=False, scoring=None, class_weight=None, **kwargs):
        """
        RocketClassifier is recommended for up to 10k time series. 
        For a larger dataset, you can use ROCKET (in Pytorch).
        scoring = None --> defaults to accuracy.
        
        Rocket args:            
            num_kernels     : int, number of random convolutional kernels (default 10,000)
            normalize_input : boolean, whether or not to normalise the input time series per instance (default True)
            random_state    : Optional random seed (default None)

        """
        try: 
            import sktime
            from sktime.transformations.panel.rocket import Rocket
        except ImportError:
            raise("You need to install sktime to be able to use RocketClassifier")
            
        self.steps = [('rocket', Rocket(num_kernels=num_kernels, normalise=normalize_input, random_state=random_state))]
        if normalize_features:
            self.steps += [('scalar', StandardScaler(with_mean=False))]
        self.steps += [('ridgeclassifiercv', RidgeClassifierCV(alphas=alphas, scoring=scoring, class_weight=class_weight, **kwargs))]
        super().__init__(self.steps, memory=memory, verbose=verbose)
        store_attr()
        self._validate_steps()

    def __repr__(self):  
        return f'Pipeline(steps={self.steps.copy()})'

    def save(self, fname='Rocket', path='./models'):
        path = Path(path)
        filename = path/fname
        with open(f'{filename}.pkl', 'wb') as output:
            pickle.dump(self, output, pickle.HIGHEST_PROTOCOL)

In [ ]:
#|export
def load_rocket(fname='Rocket', path='./models'):
    path = Path(path)
    filename = path/fname
    with open(f'{filename}.pkl', 'rb') as input:
        output = pickle.load(input)
    return output

In [ ]:
#|export
class RocketRegressor(sklearn.pipeline.Pipeline):
    """Time series regression using ROCKET features and a linear regressor"""
    
    def __init__(self, num_kernels=10_000, normalize_input=True, random_state=None, 
                 alphas=(0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0), normalize_features=True, memory=None, verbose=False, scoring=None, **kwargs):
        """
        RocketRegressor is recommended for up to 10k time series. 
        For a larger dataset, you can use ROCKET (in Pytorch).
        scoring = None --> defaults to r2.
        
        Args:            
            num_kernels     : int, number of random convolutional kernels (default 10,000)
            normalize_input : boolean, whether or not to normalise the input time series per instance (default True)
            random_state    : Optional random seed (default None)
        """
        try: 
            import sktime
            from sktime.transformations.panel.rocket import Rocket
        except ImportError:
            raise("You need to install sktime to be able to use RocketRegressor")
            
        self.steps = [('rocket', Rocket(num_kernels=num_kernels, normalise=normalize_input, random_state=random_state))]
        if normalize_features:
            self.steps += [('scalar', StandardScaler(with_mean=False))]
        self.steps += [('ridgecv', RidgeCV(alphas=alphas, scoring=scoring, **kwargs))]
        super().__init__(self.steps, memory=memory, verbose=verbose)
        store_attr()
        self._validate_steps()


    def __repr__(self):  
        return f'Pipeline(steps={self.steps.copy()})'

    def save(self, fname='Rocket', path='./models'):
        path = Path(path)
        filename = path/fname
        with open(f'{filename}.pkl', 'wb') as output:
            pickle.dump(self, output, pickle.HIGHEST_PROTOCOL)

In [ ]:
# Test sklearn Pipeline initialization stays compatible with sklearn>=1.6
import sys, types
from contextlib import contextmanager
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin, RegressorMixin

class _FakeRocket(BaseEstimator, TransformerMixin):
    def __init__(self, **kwargs): self.kwargs = kwargs
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = np.asarray(X)
        return X.reshape(X.shape[0], -1)

class _FakeClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, **kwargs): self.kwargs = kwargs
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        return self
    def predict(self, X): return np.repeat(self.classes_[0], len(X))

class _FakeRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, **kwargs): self.kwargs = kwargs
    def fit(self, X, y):
        self.y_mean_ = float(np.mean(y))
        return self
    def predict(self, X): return np.full(len(X), self.y_mean_)

@contextmanager
def _patched_rocket_deps():
    sktime_mod = types.ModuleType('sktime')
    transformations_mod = types.ModuleType('sktime.transformations')
    panel_mod = types.ModuleType('sktime.transformations.panel')
    rocket_mod = types.ModuleType('sktime.transformations.panel.rocket')
    rocket_mod.Rocket = _FakeRocket
    sktime_mod.transformations = transformations_mod
    transformations_mod.panel = panel_mod
    panel_mod.rocket = rocket_mod
    modules = {
        'sktime': sktime_mod,
        'sktime.transformations': transformations_mod,
        'sktime.transformations.panel': panel_mod,
        'sktime.transformations.panel.rocket': rocket_mod,
    }
    old_modules = {k: sys.modules.get(k) for k in modules}
    old_classifier, old_regressor = RidgeClassifierCV, RidgeCV
    try:
        sys.modules.update(modules)
        globals()['RidgeClassifierCV'] = _FakeClassifier
        globals()['RidgeCV'] = _FakeRegressor
        yield
    finally:
        globals()['RidgeClassifierCV'] = old_classifier
        globals()['RidgeCV'] = old_regressor
        for k, v in old_modules.items():
            if v is None: sys.modules.pop(k, None)
            else: sys.modules[k] = v

X = np.arange(24, dtype='float64').reshape(6, 1, 4)
y = np.array([0, 1, 0, 1, 0, 1])
with _patched_rocket_deps():
    cls = RocketClassifier(normalize_features=False)
    test_eq(cls.transform_input, None)
    cls.fit(X, y)
    reg = RocketRegressor(normalize_features=False)
    test_eq(reg.transform_input, None)
    reg.fit(X, np.arange(len(X), dtype='float64'))

In [ ]:
#|extras
# Univariate classification with sklearn-type API
dsid = 'OliveOil'
fname = 'RocketClassifier'
X_train, y_train, X_test, y_test = get_UCR_data(dsid, Xdtype='float64')
cls = RocketClassifier()
cls.fit(X_train, y_train)
cls.save(fname)
del cls
cls = load_rocket(fname)
print(cls.score(X_test, y_test))

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


0.9


In [ ]:
#|extras
# Multivariate classification with sklearn-type API
dsid = 'NATOPS'
fname = 'RocketClassifier'
X_train, y_train, X_test, y_test = get_UCR_data(dsid, Xdtype='float64')
cls = RocketClassifier()
cls.fit(X_train, y_train)
cls.save(fname)
del cls
cls = load_rocket(fname)
print(cls.score(X_test, y_test))

0.8666666666666667


In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
#|extras
# Univariate regression with sklearn-type API
dsid = 'Covid3Month'
fname = 'RocketRegressor'
X_train, y_train, X_test, y_test = get_Monash_regression_data(dsid, Xdtype='float64')
if X_train is not None: 
    rmse_scorer = make_scorer(mean_squared_error, greater_is_better=False)
    reg = RocketRegressor(scoring=rmse_scorer)
    reg.fit(X_train, y_train)
    reg.save(fname)
    del reg
    reg = load_rocket(fname)
    y_pred = reg.predict(X_test)
    print(mean_squared_error(y_test, y_pred, squared=False))

0.03908714523468997


In [ ]:
#|extras
# Multivariate regression with sklearn-type API
dsid = 'AppliancesEnergy'
fname = 'RocketRegressor'
X_train, y_train, X_test, y_test = get_Monash_regression_data(dsid, Xdtype='float64')
if X_train is not None: 
    rmse_scorer = make_scorer(mean_squared_error, greater_is_better=False)
    reg = RocketRegressor(scoring=rmse_scorer)
    reg.fit(X_train, y_train)
    reg.save(fname)
    del reg
    reg = load_rocket(fname)
    y_pred = reg.predict(X_test)
    print(mean_squared_error(y_test, y_pred, squared=False))

2.287302226812576


In [ ]:
#|eval: false
#|hide
from tsai.export import get_nb_name; nb_name = get_nb_name(locals())
from tsai.imports import create_scripts; create_scripts(nb_name)

<IPython.core.display.Javascript object>

/Users/nacho/notebooks/tsai/nbs/053_models.ROCKET.ipynb saved at 2023-04-01 19:54:02
Correct notebook to script conversion! 😃
Saturday 01/04/23 19:54:04 CEST
